# лабораторная работа №7

первая нейронная сеть: бинарные входы, обучение и интерпретация

цель работы: научиться создавать простую нейронную сеть, обучать ее и проверять результаты.

## задание 1. представление данных

число 5 записывается как 0101, поэтому для него нужны 4 входа. всего есть 4 группы, их можно записать двумя битами: 00, 01, 10, 11. число 13 записывается как 1101. остаток от деления 13 на 4 равен 1, правильный выход 01.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras

def to_bits(number, width):
    return [int(bit) for bit in format(number, f"0{width}b")]

print(to_bits(5, 4))
print(to_bits(2, 2))
print("13:", to_bits(13, 4), "ответ:", to_bits(13 % 4, 2))

## задание 2. обучающие данные

один обучающий пример это число в виде 4 битов и правильный ответ в виде 2 битов.

In [ ]:
X = []
y = []

for number in range(16):
    X.append(to_bits(number, 4))
    group = number % 4
    y.append(to_bits(group, 2))

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.float32)

print("размер X:", X.shape)
print("размер y:", y.shape)
for i in range(16):
    print(i, X[i].astype(int), "->", y[i].astype(int))

## задание 3. создание сети

у сети 4 входа и 2 выхода. скрытого слоя нет. число параметров равно 4 * 2 + 2 = 10. это 8 весов и 2 смещения.

In [ ]:
keras.utils.set_random_seed(42)
model = keras.Sequential([
    keras.Input(shape=(4,)),
    keras.layers.Dense(2, activation="sigmoid")
])
model.summary()

## задание 4. обучение

во время обучения сеть меняет веса и смещения, чтобы уменьшить ошибки. уменьшение loss означает, что ответы сети становятся ближе к правильным.

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.05),
    loss="binary_crossentropy",
    metrics=[keras.metrics.BinaryAccuracy(name="bit_accuracy")]
)

history = model.fit(X, y, epochs=200, batch_size=4, verbose=0)
print("начальный loss:", round(history.history["loss"][0], 4))
print("конечный loss:", round(history.history["loss"][-1], 4))

## задание 5. графики обучения

на первом графике видно, как уменьшается ошибка. на втором видно, как растет точность отдельных битов.

In [ ]:
plt.plot(history.history["loss"])
plt.xlabel("эпоха")
plt.ylabel("loss")
plt.title("функция потерь")
plt.show()

plt.plot(history.history["bit_accuracy"])
plt.xlabel("эпоха")
plt.ylabel("точность")
plt.title("точность по битам")
plt.show()

losses = history.history["loss"]
first_small_change = next((i + 1 for i in range(1, len(losses)) if abs(losses[i] - losses[i - 1]) < 0.001), None)
print("первая эпоха с изменением loss меньше 0.001:", first_small_change)
print("к концу обучения изменение обычно становится меньше, так как сеть уже нашла подходящие веса")

## задание 6. прогнозы

sigmoid выдает числа от 0 до 1. если значение не меньше 0.5, получаем 1, иначе 0. чем дальше оба значения от 0.5, тем увереннее решение сети.

In [ ]:
prob = model.predict(X, verbose=0)
pred_bits = (prob >= 0.5).astype(int)

for i in range(16):
    print(i, "выход:", np.round(prob[i], 3), "прогноз:", pred_bits[i], "ответ:", y[i].astype(int))

confidence = np.min(np.abs(prob - 0.5), axis=1)
best = np.argmax(confidence)
worst = np.argmin(confidence)
print("самый уверенный пример:", best, np.round(prob[best], 3))
print("наименее уверенный пример:", worst, np.round(prob[worst], 3))

## задание 7. проверка точности

точность по битам проверяет каждый выход отдельно. точность по группам требует, чтобы сразу оба бита были правильными. поэтому второй способ строже.

In [ ]:
bit_accuracy = np.mean(pred_bits == y.astype(int))
exact_matches = np.all(pred_bits == y.astype(int), axis=1)
exact_accuracy = exact_matches.mean()
errors = np.where(~exact_matches)[0]
print("точность по битам:", round(bit_accuracy, 4))
print("точность по группам:", round(exact_accuracy, 4))
print("числа с ошибками:", errors.tolist())

## задание 8. веса и смещения

веса показывают влияние входных битов на каждый выход. самые важные для правила остатка от деления на 4 это два последних бита числа.

In [ ]:
weights, biases = model.layers[0].get_weights()
print("веса:")
print(np.round(weights, 3))
print("смещения:", np.round(biases, 3))

for output in range(2):
    indexes = np.argsort(np.abs(weights[:, output]))[-2:][::-1]
    print("выход", output + 1, "самые большие веса у входов", (indexes + 1).tolist())
    for index in indexes:
        print("бит", index + 1, "вес", round(float(weights[index, output]), 3))

## задание 9. ручная проверка

операция @ умножает входные значения на веса. затем добавляется смещение. sigmoid переводит результат в диапазон от 0 до 1.

In [ ]:
x = X[5]
z = x @ weights + biases
manual = 1 / (1 + np.exp(-z))
auto = model.predict(x.reshape(1, -1), verbose=0)[0]
print("ручной расчет:", manual)
print("ответ модели:", auto)
print("совпадают:", np.allclose(manual, auto))

## задание 10. эксперимент

изменим только число эпох с 200 на 20. остальные настройки оставим прежними. начальные веса будут одинаковыми благодаря фиксированному случайному числу.

In [ ]:
keras.utils.set_random_seed(42)
model2 = keras.Sequential([
    keras.Input(shape=(4,)),
    keras.layers.Dense(2, activation="sigmoid")
])
model2.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.05),
    loss="binary_crossentropy",
    metrics=[keras.metrics.BinaryAccuracy(name="bit_accuracy")]
)
history2 = model2.fit(X, y, epochs=20, batch_size=4, verbose=0)
prob2 = model2.predict(X, verbose=0)
pred2 = (prob2 >= 0.5).astype(int)
exact2 = np.all(pred2 == y.astype(int), axis=1).mean()
print("200 эпох: loss", round(history.history["loss"][-1], 4), "точность по битам", round(bit_accuracy, 4), "по группам", round(exact_accuracy, 4))
print("20 эпох: loss", round(history2.history["loss"][-1], 4), "точность по битам", round(float(np.mean(pred2 == y.astype(int))), 4), "по группам", round(float(exact2), 4))
plt.plot(history.history["loss"], label="200 эпох")
plt.plot(history2.history["loss"], label="20 эпох")
plt.xlabel("эпоха")
plt.ylabel("loss")
plt.title("сравнение обучения")
plt.legend()
plt.show()

при 20 эпохах сеть делает меньше шагов обучения. поэтому веса успевают измениться меньше и ошибка обычно остается выше. при 200 эпохах у сети больше времени, чтобы подобрать веса и повысить точность.

## вывод

в этой работе я создала простую нейронную сеть с четырьмя входами и двумя выходами. сеть училась определять группу числа по его двоичной записи. во время обучения менялись веса и смещения, уменьшалась ошибка. я проверила прогнозы, точность, веса и ручной расчет, а также сравнила обучение за 200 и 20 эпох.

## контрольные вопросы

1. вход сети это 4 бита числа от 0 до 15.
2. две двоичные цифры позволяют записать 4 группы.
3. вес показывает, насколько сильно вход влияет на нейрон.
4. смещение помогает нейрону менять результат даже при нулевых входах.
5. sigmoid переводит число в значение от 0 до 1.
6. эпоха это один проход по всем обучающим примерам.
7. функция потерь показывает величину ошибки сети.
8. оптимизатор изменяет веса и смещения для уменьшения ошибки.
9. порог нужен, чтобы получить 0 или 1 из выхода sigmoid.
10. начальные веса могут быть случайными, поэтому итоговые значения отличаются.
11. точность по битам проверяет каждый бит, а точность группы проверяет оба сразу.
12. маленькую сеть легко разобрать, но она не решает сложные задачи, как современные нейросети.